## 嵌入类

In [ ]:
import os
import numpy as np
from openai import OpenAI
from langchain_core.embeddings import Embeddings

# 设置阿里云API密钥
API_KEY = os.getenv("DASHSCOPE_API_KEY")
if not API_KEY:
    raise ValueError("请设置环境变量 DASHSCOPE_API_KEY")
BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "text-embedding-v4"

# 初始化OpenAI客户端
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

class AliyunEmbeddings(Embeddings):
    """阿里云文本嵌入模型，返回L2归一化后的向量（便于余弦相似度计算）"""
    def __init__(self, client, model_name="text-embedding-v4", batch_size=10):
        self.client = client
        self.model_name = model_name
        self.batch_size = batch_size

    def _normalize(self, vec):
        """L2归一化"""
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def embed_documents(self, texts):
        all_embeddings = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i+self.batch_size]
            resp = self.client.embeddings.create(model=self.model_name, input=batch)
            batch_embeddings = [self._normalize(np.array(item.embedding)) for item in resp.data]
            all_embeddings.extend(batch_embeddings)
        return all_embeddings

    def embed_query(self, text):
        resp = self.client.embeddings.create(model=self.model_name, input=text)
        vec = np.array(resp.data[0].embedding)
        return self._normalize(vec).tolist()

# 实例化嵌入模型（构建和检索均使用此实例）
aliyun_emb = AliyunEmbeddings(client, model_name=MODEL_NAME, batch_size=10)

## 混合检索参考

In [ ]:
import math
import numpy as np

def bm25_score(query_words, doc_id, inverted_index, doc_lengths, N, avg_len, k1=1.5, b=0.75):
    """
    计算单个文档的BM25得分（标准实现）
    :param query_words: 查询分词列表
    :param doc_id: 文档ID
    :param inverted_index: 倒排索引 {词: {doc_id: 词频}}
    :param doc_lengths: 文档长度 {doc_id: 词数}
    :param N: 文档总数
    :param avg_len: 平均文档长度
    :param k1, b: BM25参数
    :return: BM25得分
    """
    score = 0.0
    doc_len = doc_lengths[doc_id]
    for w in query_words:
        if w not in inverted_index or doc_id not in inverted_index[w]:
            continue
        freq = inverted_index[w][doc_id]
        # 逆文档频率 (IDF)
        n_w = len(inverted_index[w])  # 包含词w的文档数
        idf = math.log((N - n_w + 0.5) / (n_w + 0.5) + 1)
        # 词频因子 (TF)
        tf_part = freq * (k1 + 1) / (freq + k1 * (1 - b + b * (doc_len / avg_len)))
        score += idf * tf_part
    return score

def hybrid_search(query, k_vector=8, k_final=3, alpha=0.5):
    """
    两阶段混合检索：
    1. 向量召回：使用FAISS（L2距离）获取 Top-(k_vector*2) 候选文档
    2. 精确计算余弦相似度（基于保存的归一化向量）和BM25得分，加权融合后取Top-k_final
    :param query: 用户查询字符串
    :param k_vector: 第一阶段向量召回的数量（实际多取一倍以确保覆盖）
    :param k_final: 最终返回的文档数
    :param alpha: 余弦相似度的权重，BM25权重为 1-alpha
    :return: 列表，每个元素为 (Document对象, 综合得分)
    """
    # 1. 查询向量化（已归一化）
    query_vec = aliyun_emb.embed_query(query)  # 返回归一化向量

    # 2. 使用FAISS进行第一阶段召回（基于L2距离）
    #   注意：vectorstore 是基于L2距离的IndexFlatL2，返回的score是L2距离（越小越相似）
    #   因为向量已归一化，L2距离与余弦相似度单调负相关，所以用L2召回不会漏掉最相似文档
    docs_and_scores = vectorstore.similarity_search_with_score_by_vector(
        query_vec, k=k_vector * 2
    )
    candidate_docs = [doc for doc, _ in docs_and_scores]  # 候选文档列表

    # 3. 提取候选文档ID并计算精确的余弦相似度（利用保存的向量数组）
    cos_scores = []
    for doc in candidate_docs:
        doc_id = doc.metadata['UNICODE']
        idx = doc_id_to_index[doc_id]          # 文档ID在向量数组中的索引
        doc_vec = doc_embeddings[idx]          # 已归一化
        cos_sim = np.dot(query_vec, doc_vec)   # 归一化后点积即余弦相似度
        cos_scores.append(cos_sim)

    # 4. 对候选文档计算BM25得分
    query_words = list(jieba.cut(query))
    bm25_scores = []
    for doc in candidate_docs:
        doc_id = doc.metadata['UNICODE']
        bm25 = bm25_score(query_words, doc_id, inverted_index, doc_lengths, N, avg_len)
        bm25_scores.append(bm25)

    # 5. 得分融合
    cos_scores = np.array(cos_scores)
    bm25_scores = np.array(bm25_scores)

    # 将BM25得分归一化到[0,1]区间
    bm25_min, bm25_max = bm25_scores.min(), bm25_scores.max()
    if bm25_max - bm25_min > 1e-9:
        bm25_norm = (bm25_scores - bm25_min) / (bm25_max - bm25_min)
    else:
        bm25_norm = np.zeros_like(bm25_scores)

    # 余弦相似度截断到[0,1]（理论上归一化向量点积在[-1,1]，但检索场景多为正，保留非负）
    cos_scores = np.clip(cos_scores, 0, 1)

    # 加权融合
    hybrid_scores = alpha * cos_scores + (1 - alpha) * bm25_norm

    # 6. 按综合得分降序排序，取前k_final
    sorted_indices = np.argsort(hybrid_scores)[::-1]
    results = []
    for idx in sorted_indices[:k_final]:
        doc = candidate_docs[idx]
        score = hybrid_scores[idx]
        results.append((doc, score))

    return results

## 示例

In [ ]:
# 示例使用
if __name__ == "__main__":
    query = "木+伏"
    top_docs = hybrid_search(query)
    print(f"查询: {query}\n")
    for i, (doc, score) in enumerate(top_docs):
        print(f"Top {i+1} (得分: {score:.4f}):")
        print(f"内容: {doc.page_content[:200]}...")
        print(f"元数据: {doc.metadata}\n")